In [1]:
import sys; sys.path.append("..")
import pandas as pd
import plotly.express as px

news = pd.read_parquet("../data/processed/news.parquet")
news["date"] = pd.to_datetime(news["date"])
monthly = news.groupby([pd.Grouper(key="date", freq="ME"), "ticker"]).size().reset_index(name="n")
fig = px.line(monthly, x="date", y="n", color="ticker", title="News volume per month")
fig.show()

In [2]:
reddit = pd.read_parquet("../data/processed/reddit.parquet")
lengths = reddit["text"].str.split().str.len()
print(lengths.describe().round(1))
print("posts under 10 words:", (lengths < 10).mean().round(2))

count    4072.0
mean      628.6
std       782.5
min         1.0
25%       117.0
50%       354.0
75%       866.2
max      6442.0
Name: text, dtype: float64
posts under 10 words: 0.06


In [3]:
import re
from collections import Counter

STOP = set(("the a an and or of to in for on is are was were be been with as at by it its "
            "this that from we our their they he she you i will would could should have has "
            "had not no but if then than so what which who all any").split())

def top_words(texts, n=12):
    c = Counter()
    for t in texts:
        c.update(w for w in re.findall(r"[a-z']+", str(t).lower())
                 if w not in STOP and len(w) > 2)
    return c.most_common(n)

sec_text = open("../data/processed/sec/NVDA_10K_2026-02-25.txt", encoding="utf-8").read()

print("SEC:   ", top_words([sec_text[:100000]]))
print("News:  ", top_words(news["headline"].sample(3000, random_state=1)))
print("Reddit:", top_words(reddit["text"].sample(3000, random_state=1)))

SEC:    [('may', 130), ('products', 125), ('nvidia', 78), ('software', 68), ('data', 67), ('business', 66), ('customers', 59), ('including', 58), ('financial', 57), ('other', 56), ('new', 54), ('product', 54)]
News:   [('johnson', 342), ('nvidia', 297), ('stocks', 287), ('earnings', 259), ('shares', 240), ('market', 232), ('tesla', 219), ('price', 178), ('target', 174), ('stock', 166), ('trading', 155), ('says', 154)]
Reddit: [('market', 6620), ('stock', 6087), ('more', 5984), ('can', 5514), ('company', 5509), ('like', 5004), ('just', 4563), ('about', 4559), ('some', 4395), ('price', 4383), ('out', 4213), ('there', 4111)]


## 📰 Findings — Text EDA (Week 1 close)
*(Written by mentor during illness accommodation, Jul 22 — computed from our actual data. Review when recovered.)*

**1. News coverage explodes over time.** Yearly volume: 914 (2017) → 1,736 (2018) → 4,406 (2019) → 4,423 (2020, partial year). Peak month: **May 2020, 1,170 headlines** (COVID-crash aftermath), vs a median month of just 49. Sentiment analysis will have rich data post-2019 and sparse data before — trend charts must not over-read the thin early years.

**2. Reddit posts are LONGER than expected — a corrected assumption.** Median post: **354 words**; only **6% are under 10 words**. The expectation was meme-length one-liners; this dataset actually skews toward long "due diligence" essays. Good news for Day 12: plenty of text per post for a sentiment model to chew on. (Recorded as a corrected prediction — evidence beat expectation.)

**3. Three corpora, three different languages.**
- **SEC filings:** products, software, business, computing — formal, self-describing, no emotion
- **News headlines:** earnings, shares, market, price — market-centric, event-driven
- **Reddit:** market, stock, like, just, can — casual, generic, opinion-heavy
A sentiment model trained on movie reviews would be lost here; even across our own corpora the registers differ sharply. This is the concrete justification for **FinBERT** (finance-trained) on Day 12.

**4. Known gaps, documented:** MSFT has zero news rows (never collected by the source); news data ends mid-2020; Reddit covers only Jan–Aug 2021 (meme-stock era — uniquely loud sentiment). All three caveats travel with any conclusion drawn from these datasets.

**Week 1 data layer: COMPLETE.** Six sources, six modules, one command (`run_all.py`), every dataset understood — strengths, quirks, and gaps.
